In [1]:
# ============================================
# 1) Imports
# ============================================
import pandas as pd
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

In [2]:
# ============================================
# 2) Load dataset
# ============================================
DATA_PATH = "balanced_network_data.csv"

df = pd.read_csv(DATA_PATH)

print("Loaded shape:", df.shape)
print(df["Label"].value_counts().head(20))

Loaded shape: (6870587, 23)
Label
Benign                      4122352
DDOS attack-HOIC             686012
DDoS attacks-LOIC-HTTP       576191
DoS attacks-Hulk             461912
Bot                          286191
FTP-BruteForce               193360
SSH-Bruteforce               187589
Infilteration                161934
DoS attacks-SlowHTTPTest     139890
DoS attacks-GoldenEye         41508
DoS attacks-Slowloris         10990
DDOS attack-LOIC-UDP           1730
Brute Force -Web                611
Brute Force -XSS                230
SQL Injection                    87
Name: count, dtype: int64


In [3]:
# ============================================
# 3) Selected network features
# ============================================
network_features = [
    "Dst Port",
    "Protocol",
    "Flow Duration",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Max",
    "Fwd IAT Min",
    "Fwd IAT Tot",
    "Flow Pkts/s",
    "Fwd Pkts/s",
    "Bwd Pkts/s",
    "Tot Fwd Pkts",
    "Subflow Fwd Pkts",
    "Fwd Header Len",
    "Bwd Header Len",
    "Init Fwd Win Byts",
    "Init Bwd Win Byts",
    "PSH Flag Cnt",
    "RST Flag Cnt",
    "ACK Flag Cnt",
    "ECE Flag Cnt"
]

missing = [c for c in network_features if c not in df.columns]
if missing:
    print("Missing features skipped:", missing)

network_features = [c for c in network_features if c in df.columns]
print("Using features:", len(network_features))

Using features: 22


In [4]:
# ============================================
# 4) Clean data
# ============================================
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["Label"]).copy()

X = df[network_features].copy()
y = df["Label"].astype(str).copy()

for col in X.columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

mask = X.notna().all(axis=1)
X = X[mask]
y = y[mask]

# Binary labels:
# 0 = Benign
# 1 = Attack
y_binary = (y != "Benign").astype(int)

print("Cleaned X shape:", X.shape)
print("Binary label counts:")
print(y_binary.value_counts())

Cleaned X shape: (6838750, 22)
Binary label counts:
Label
0    4091816
1    2746934
Name: count, dtype: int64


In [5]:
# ============================================
# 5) Train-test split
# ============================================
# stratify keeps benign/attack ratio similar in train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (5471000, 22)
Test shape: (1367750, 22)


In [6]:
# ============================================
# 6) Optional scaling
# ============================================
# Random Forest does not require scaling, but keeping it
# here for consistency with your previous pipeline.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [8]:
# ============================================
# 7) Reduce training size for from-scratch RF
# ============================================
# VERY IMPORTANT: pure Python RF will be too slow on millions of rows
MAX_TRAIN_SAMPLES = 50000   # change to 10000 / 20000 / 50000 depending on your machine
MAX_TEST_SAMPLES = 20000

rng = np.random.RandomState(42)

if len(X_train) > MAX_TRAIN_SAMPLES:
    idx = rng.choice(len(X_train), size=MAX_TRAIN_SAMPLES, replace=False)
    X_train_small = X_train[idx]
    y_train_small = y_train.iloc[idx].to_numpy()
else:
    X_train_small = X_train
    y_train_small = y_train.to_numpy()

if len(X_test) > MAX_TEST_SAMPLES:
    idx = rng.choice(len(X_test), size=MAX_TEST_SAMPLES, replace=False)
    X_test_small = X_test[idx]
    y_test_small = y_test.iloc[idx].to_numpy()
else:
    X_test_small = X_test
    y_test_small = y_test.to_numpy()

print("Training subset:", X_train_small.shape)
print("Testing subset:", X_test_small.shape)

Training subset: (50000, 22)
Testing subset: (20000, 22)


In [9]:
# ============================================
# 8) Decision Tree from scratch
# ============================================
class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value   # leaf value


class DecisionTreeScratch:
    def __init__(self, max_depth=10, min_samples_split=2, max_features=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.root = None

    def fit(self, X, y):
        self.n_features_ = X.shape[1]
        self.root = self._grow_tree(X, y, depth=0)

    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _gini(self, y):
        classes, counts = np.unique(y, return_counts=True)
        probs = counts / counts.sum()
        return 1.0 - np.sum(probs ** 2)

    def _best_split(self, X, y, feature_idxs):
        best_gain = -1
        split_idx, split_threshold = None, None

        parent_gini = self._gini(y)
        n = len(y)

        for feature in feature_idxs:
            values = X[:, feature]
            thresholds = np.unique(values)

            # To reduce time, sample thresholds if too many unique values
            if len(thresholds) > 50:
                thresholds = np.percentile(thresholds, np.linspace(5, 95, 20))

            for threshold in thresholds:
                left_mask = values <= threshold
                right_mask = values > threshold

                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                gini_left = self._gini(y_left)
                gini_right = self._gini(y_right)

                weighted_gini = (
                    (len(y_left) / n) * gini_left +
                    (len(y_right) / n) * gini_right
                )

                gain = parent_gini - weighted_gini

                if gain > best_gain:
                    best_gain = gain
                    split_idx = feature
                    split_threshold = threshold

        return split_idx, split_threshold

    def _most_common_label(self, y):
        return Counter(y).most_common(1)[0][0]

    def _grow_tree(self, X, y, depth):
        num_samples, num_features = X.shape
        num_labels = len(np.unique(y))

        # stopping conditions
        if (
            depth >= self.max_depth or
            num_labels == 1 or
            num_samples < self.min_samples_split
        ):
            leaf_value = self._most_common_label(y)
            return DecisionTreeNode(value=leaf_value)

        # random subset of features
        feature_idxs = np.arange(num_features)
        if self.max_features is not None:
            feature_idxs = np.random.choice(
                num_features, self.max_features, replace=False
            )

        best_feature, best_threshold = self._best_split(X, y, feature_idxs)

        if best_feature is None:
            leaf_value = self._most_common_label(y)
            return DecisionTreeNode(value=leaf_value)

        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold

        left_child = self._grow_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._grow_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(
            feature=best_feature,
            threshold=best_threshold,
            left=left_child,
            right=right_child
        )

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

In [10]:
# ============================================
# 9) Random Forest from scratch
# ============================================
class RandomForestScratch:
    def __init__(self, n_trees=10, max_depth=10, min_samples_split=2, max_features="sqrt"):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.trees = []

    def _get_max_features(self, n_features):
        if self.max_features == "sqrt":
            return max(1, int(np.sqrt(n_features)))
        elif self.max_features == "log2":
            return max(1, int(np.log2(n_features)))
        elif isinstance(self.max_features, int):
            return self.max_features
        else:
            return n_features

    def _bootstrap_sample(self, X, y):
        n_samples = X.shape[0]
        idxs = np.random.choice(n_samples, n_samples, replace=True)
        return X[idxs], y[idxs]

    def fit(self, X, y):
        self.trees = []
        n_features = X.shape[1]
        max_feats = self._get_max_features(n_features)

        for i in range(self.n_trees):
            print(f"Training tree {i+1}/{self.n_trees}...")
            tree = DecisionTreeScratch(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=max_feats
            )
            X_sample, y_sample = self._bootstrap_sample(X, y)
            tree.fit(X_sample, y_sample)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])  # shape: [n_trees, n_samples]
        tree_preds = np.swapaxes(tree_preds, 0, 1)  # shape: [n_samples, n_trees]

        y_pred = []
        for preds in tree_preds:
            y_pred.append(Counter(preds).most_common(1)[0][0])
        return np.array(y_pred)

    def predict_proba_attack(self, X):
        """
        Returns probability of class 1 (attack) based on tree voting fraction.
        """
        tree_preds = np.array([tree.predict(X) for tree in self.trees])  # [n_trees, n_samples]
        attack_prob = np.mean(tree_preds == 1, axis=0)
        return attack_prob

In [ ]:
# ============================================
# 10) Train from-scratch Random Forest
# ============================================
rf = RandomForestScratch(
    n_trees=10,           # increase to 20 if machine can handle
    max_depth=10,         # can try 12 or 15
    min_samples_split=5,
    max_features="sqrt"
)

rf.fit(X_train_small, y_train_small)

Training tree 1/10...
Training tree 2/10...
Training tree 3/10...
Training tree 4/10...
Training tree 5/10...
Training tree 6/10...
Training tree 7/10...
Training tree 8/10...
Training tree 9/10...
Training tree 10/10...


In [19]:
# ============================================
# 11) Predict
# ============================================
y_pred = rf.predict(X_test_small)
y_scores = rf.predict_proba_attack(X_test_small)

In [13]:
# ============================================
# 12) Evaluate
# ============================================
cm = confusion_matrix(y_test_small, y_pred, labels=[0, 1])
print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test_small, y_pred, target_names=["Benign", "Attack"]))

auc = roc_auc_score(y_test_small, y_scores)
print("ROC-AUC:", auc)


Confusion Matrix [[TN, FP], [FN, TP]]:
[[11873   112]
 [  468  7547]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.98     11985
      Attack       0.99      0.94      0.96      8015

    accuracy                           0.97     20000
   macro avg       0.97      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000

ROC-AUC: 0.9754774201792581


In [23]:
# ============================================
# 10) Train from-scratch Random Forest
# ============================================
rf = RandomForestScratch(
    n_trees=20,           # increase to 20 if machine can handle
    max_depth=15,         # can try 12 or 15
    min_samples_split=5,
    max_features="sqrt"
)

rf.fit(X_train_small, y_train_small)

# download the model
import joblib
joblib.dump(rf, "random_forest_scratch.pkl")

Training tree 1/20...
Training tree 2/20...
Training tree 3/20...
Training tree 4/20...
Training tree 5/20...
Training tree 6/20...
Training tree 7/20...
Training tree 8/20...
Training tree 9/20...
Training tree 10/20...
Training tree 11/20...
Training tree 12/20...
Training tree 13/20...
Training tree 14/20...
Training tree 15/20...
Training tree 16/20...
Training tree 17/20...
Training tree 18/20...
Training tree 19/20...
Training tree 20/20...


['random_forest_scratch.pkl']

In [ ]:
# ============================================
# 11) Predict
# ============================================
y_pred = rf.predict(X_test_small)
y_scores = rf.predict_proba_attack(X_test_small)

In [20]:
# ============================================
# 12) Evaluate
# ============================================
cm = confusion_matrix(y_test_small, y_pred, labels=[0, 1])
print("\nConfusion Matrix [[TN, FP], [FN, TP]]:")
print(cm)

print("\nClassification Report:")
print(classification_report(y_test_small, y_pred, target_names=["Benign", "Attack"]))

auc = roc_auc_score(y_test_small, y_scores)
print("ROC-AUC:", auc)


Confusion Matrix [[TN, FP], [FN, TP]]:
[[11920    65]
 [  465  7550]]

Classification Report:
              precision    recall  f1-score   support

      Benign       0.96      0.99      0.98     11985
      Attack       0.99      0.94      0.97      8015

    accuracy                           0.97     20000
   macro avg       0.98      0.97      0.97     20000
weighted avg       0.97      0.97      0.97     20000

ROC-AUC: 0.9801376694875665


In [14]:
# ============================================
# 13) Per-attack detection rate
# ============================================
# For per-attack evaluation, use original labels from df
# rebuild same split indices logic on cleaned dataset
X_all = X.to_numpy()
y_all_binary = y_binary.to_numpy()
y_all_labels = y.to_numpy()

# make a matching train-test split with indices
indices = np.arange(len(X))
train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

# if test subset was sampled, track that
if len(test_idx) > MAX_TEST_SAMPLES:
    sampled_test_local = idx
    test_idx_small = test_idx[sampled_test_local]
else:
    test_idx_small = test_idx

test_labels_original = y_all_labels[test_idx_small]

attack_names = [lab for lab in np.unique(test_labels_original) if lab != "Benign"]

rows = []
for attack_name in attack_names:
    mask_attack = (test_labels_original == attack_name)
    if mask_attack.sum() == 0:
        continue

    detected_rate = (y_pred[mask_attack] == 1).mean()
    rows.append((attack_name, mask_attack.sum(), detected_rate))

report_df = pd.DataFrame(rows, columns=["Attack Label", "Samples", "Detection Rate"])
report_df = report_df.sort_values("Detection Rate", ascending=False)

print("\nPer-attack detection rate:")
print(report_df)


Per-attack detection rate:
                Attack Label  Samples  Detection Rate
1           Brute Force -XSS        1        1.000000
4     DDoS attacks-LOIC-HTTP     1660        1.000000
3       DDOS attack-LOIC-UDP       10        1.000000
6           DoS attacks-Hulk     1355        1.000000
5      DoS attacks-GoldenEye      108        1.000000
9             FTP-BruteForce      564        1.000000
8      DoS attacks-Slowloris       28        1.000000
7   DoS attacks-SlowHTTPTest      404        1.000000
2           DDOS attack-HOIC     1976        0.999494
12            SSH-Bruteforce      568        0.998239
0                        Bot      847        0.996458
10             Infilteration      493        0.062880
11             SQL Injection        1        0.000000
